In [0]:
from pyspark import pipelines as dp
from trip_transforms import aggregate_daily_route_metrics, normalize_trips

# Create a unified Gold streaming table
dp.create_streaming_table(
    name="all_taxi_trips",
    comment="Unified Yellow and Green taxi trips from Silver",
    table_properties={"delta.feature.timestampNtz": "supported"}
)

# Append Yellow Taxi records
@dp.append_flow(target="all_taxi_trips", name="yellow_taxi_flow")
def yellow_taxi_flow():
    return normalize_trips(
        spark.readStream.table("silver.yellow_trips"),
        taxi_type="yellow",
        pickup_column="tpep_pickup_datetime",
        dropoff_column="tpep_dropoff_datetime"
    )

# Append Green Taxi records
@dp.append_flow(target="all_taxi_trips", name="green_taxi_flow")
def green_taxi_flow():
    return normalize_trips(
        spark.readStream.table("silver.green_trips"),
        taxi_type="green",
        pickup_column="lpep_pickup_datetime",
        dropoff_column="lpep_dropoff_datetime"
    )




In [0]:
# Create dashboard-ready route metrics
@dp.materialized_view(
    name="daily_route_metrics",
    comment="Daily taxi demand, revenue, route and payment statistics"
)
def daily_route_metrics():
    trips_df = spark.read.table("all_taxi_trips")
    taxi_zones_df = spark.read.table("bronze.taxi_zones_raw")
    return aggregate_daily_route_metrics(trips_df, taxi_zones_df)

